# RAG in 5 Minutes

**Goal:** Answer questions about your documents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

In [ ]:
# Setup (run once)
# !pip install anthropic chromadb

In [ ]:
# Quick Win: Working RAG in one cell!

import anthropic
import chromadb

# 1. Create vector store
chroma = chromadb.Client()
collection = chroma.create_collection("docs")

# 2. Add your documents (these are examples - use your own!)
docs = [
    "The company was founded in 2020 by Jane Smith.",
    "Our main product is CloudSync, a data synchronization tool.",
    "We have 50 employees across 3 offices in NYC, London, and Tokyo.",
    "Revenue in 2024 was $10 million, up 40% from 2023.",
]
collection.add(documents=docs, ids=[f"doc_{i}" for i in range(len(docs))])

# 3. Query and answer
def ask(question: str) -> str:
    # Find relevant docs
    results = collection.query(query_texts=[question], n_results=2)
    context = "\n".join(results["documents"][0])
    
    # Ask LLM
    client = anthropic.Anthropic()
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]
    )
    return response.content[0].text

# Try it!
print(ask("Who founded the company?"))
print(ask("What was the revenue growth?"))

## How It Works

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│  Question   │───▶│   Search    │───▶│   Context   │
│ "Who founded│    │  Vector DB  │    │ + Question  │
│  company?"  │    │             │    │             │
└─────────────┘    └─────────────┘    └──────┬──────┘
                                             │
                                             ▼
┌─────────────┐                       ┌─────────────┐
│   Answer    │◀──────────────────────│     LLM     │
│"Jane Smith" │                       │             │
└─────────────┘                       └─────────────┘
```

RAG = Retrieve relevant docs → Augment prompt with context → Generate answer

In [ ]:
# Modify This: Try different chunk sizes and see the difference

def chunk_text(text: str, chunk_size: int = 200) -> list:
    """Split text into chunks. Try: 100, 200, 500"""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks

# Example: chunk a longer document
long_doc = """Your long document here. " "" * 50  # Simulate long text
chunks = chunk_text(long_doc, chunk_size=100)  # TODO: Try 50, 100, 200
print(f"Created {len(chunks)} chunks")

## Copy-Paste Template

Production-ready RAG class:

In [ ]:
# Copy this entire cell for your projects

import anthropic
import chromadb
from pathlib import Path

class SimpleRAG:
    def __init__(self, collection_name: str = "documents"):
        self.client = anthropic.Anthropic()
        self.chroma = chromadb.PersistentClient(path="./chroma_db")  # Persists to disk
        self.collection = self.chroma.get_or_create_collection(collection_name)
    
    def add_documents(self, docs: list[str], ids: list[str] = None):
        if ids is None:
            ids = [f"doc_{i}" for i in range(len(docs))]
        self.collection.add(documents=docs, ids=ids)
    
    def add_file(self, filepath: str):
        text = Path(filepath).read_text()
        chunks = [text[i:i+1000] for i in range(0, len(text), 800)]  # Overlapping chunks
        self.add_documents(chunks, [f"{filepath}_{i}" for i in range(len(chunks))])
    
    def ask(self, question: str, n_results: int = 3) -> str:
        results = self.collection.query(query_texts=[question], n_results=n_results)
        context = "\n---\n".join(results["documents"][0])
        
        response = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            system="Answer based only on the provided context. If unsure, say so.",
            messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]
        )
        return response.content[0].text

# Usage
# rag = SimpleRAG()
# rag.add_file("my_document.txt")
# print(rag.ask("What is this document about?"))

## What's Next?

- **Add Tools:** [02_agent_with_tools.ipynb](02_agent_with_tools.ipynb) - Combine RAG with actions
- **Go Deeper:** [../concepts/visual_guides/rag_pipeline.md](../concepts/visual_guides/rag_pipeline.md)
- **Production Template:** [../templates/rag_template.py](../templates/rag_template.py)